### Load the data
to make it easier, we are only going to work with the english dataset

In [28]:
import pandas as pd

classes_en = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}
train_en = pd.read_csv("https://raw.githubusercontent.com/Samkas/TextMiningWS25/refs/heads/main/data/AGNews/train.csv", 
                       names = ["Label", "Title", "Article"],
                       encoding = "utf-8")
test_en = pd.read_csv("https://raw.githubusercontent.com/Samkas/TextMiningWS25/refs/heads/main/data/AGNews/test.csv", 
                      names = ["Label", "Title", "Article"],
                      encoding = "utf-8")

sample = train_en.sample(5000)
labels = [classes_en[int(row["Label"])] for i, row in sample.iterrows()]
docs = [row["Article"] for i, row in sample.iterrows()]

In [29]:
import re

# Function to clean text using regex
# Removes non-alphabetic characters, optionally keeping sentence-ending punctuation
def clean(text: str, keep_sentences: bool = True) -> str:
    if keep_sentences:
        return re.sub(r"[^a-zA-Z\.\?\!\s]+", "", text)
    return re.sub(r"[^a-zA-Z\s]+", "", text)

In [30]:
# first, lets clean our documents
docs_cleaned = [clean(row["Article"]) for i, row in sample.iterrows()]

## Feature Representation

We already worked with TF-IDF with Gensim, for this exercise we are switching to sklearn and use it for TF-IDF

[https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html]( https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)

In [31]:
from nltk.corpus import stopwords as nltkStopwords
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# Load English stopwords from NLTK
stopwords_en = list(nltkStopwords.words("english"))

# Initialize the TF-IDF vectorizer
# - strip_accents="unicode": normalize accented characters
# - stop_words=stopwords_en: remove common English stopwords
tfidf_vectorizer = TfidfVectorizer(strip_accents="unicode", 
                                   stop_words=stopwords_en)

# Fit the vectorizer to the document collection and transform it into a TF-IDF matrix
X_tfidf = tfidf_vectorizer.fit_transform(docs)

# Get the list of feature names (i.e., vocabulary terms)
tfidf_features_names = tfidf_vectorizer.get_feature_names_out()

# Print the shape of the TF-IDF matrix: (num_documents, num_features)
print(X_tfidf.shape)

# Print the number of unique features (terms) extracted
print(tfidf_features_names.shape)

(5000, 16175)
(16175,)


In [32]:
# convert everything to a pandas df, we need to use toarray() to get a dense matrix
tfidf = pd.DataFrame(X_tfidf.toarray(), columns = tfidf_features_names)
tfidf["Label"] = labels
tfidf.to_csv("tfidf_en.csv")

In [33]:
tfidf.head()

,00,000,000m,01,02,025bil,026,0291,03,038,...,zone,zoock,zook,zorilla,zte,zurich,zvonareva,zyman,zz,Label
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,World
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Sports
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,World
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Sci/Tech
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Business


In [34]:
# doing the same for the cleaned documents

# ? your code

Alternatively, we can use a word count matrix, which consists of the absolute term 

[https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)

In [35]:
#Initialize the CountVectorizer
# - strip_accents="unicode": normalize accented characters
# - stop_words=stopwords_en: remove common English stopwords
count_vectorizer = CountVectorizer(strip_accents="unicode",
                                   stop_words=stopwords_en)

# Fit the vectorizer to the document collection and transform it into a term-frequency matrix
X_counts = count_vectorizer.fit_transform(docs)

# Get the list of feature names (i.e., vocabulary terms)
counts_features_names = count_vectorizer.get_feature_names_out()

# Print the shape of the term-frequency matrix: (num_documents, num_features)
print(X_counts.shape)

# Print the number of unique features (terms) extracted
print(counts_features_names.shape)


(5000, 16175)
(16175,)


In [36]:
# again, to df

counts = pd.DataFrame(X_counts.toarray(), columns = counts_features_names)
counts["Label"] = labels

counts.head()

,00,000,000m,01,02,025bil,026,0291,03,038,...,zone,zoock,zook,zorilla,zte,zurich,zvonareva,zyman,zz,Label
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,World
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Sports
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,World
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Sci/Tech
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Business


In [37]:
# lets to the same for the cleaned documents

# ? your code

### Finally, we can also use embeddings (Word2Vec or Doc2Vec) to represent our documents

You can train these embeddings with your own data by yourself e.g. as described here [https://www.tensorflow.org/tutorials/text/word2vec](https://www.tensorflow.org/tutorials/text/word2vec)  
or you use a pre-trained model like spaCy (remember our first exercise).

Since creating these embeddings may take some time, I did that beforehand but this is the code to do so:

```python
import spacy
import numpy as np
nlp = spacy.load("en_core_web_lg")
doc2vec = [nlp(doc).vector for doc in docs]
vectors = np.array(doc2vec)
vec_space = pd.DataFrame(vectors)
vec_space["Label"] = labels
vec_space.to_csv("doc2vec_en.csv")
```

In [38]:
# loading the document vectors

vec_space = pd.read_csv("https://raw.githubusercontent.com/Samkas/TextMiningWS25/refs/heads/main/data/Doc2Vec/doc2vec_en.csv", index_col = 0)
X_doc2vec = vec_space.loc[:, vec_space.columns != "Label"].values
Y_doc2vec = vec_space["Label"].values

print(X_doc2vec.shape)
print(Y_doc2vec.shape)

(5000, 300)
(5000,)


# Clustering

Each of the feature representations shown above can be used for clustering or classifiction. 

We are going to use the kMeans clustering and assess the quality of our clusters using silhouette plots and scores.

You should:
- Read the documentation about kMeans in sklearn [https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)
- Try out different numbers of clusters
- Try out the different feature represenations
- What works well and what doesn't work well?
- Which distance metric does sklearn's implemenation of kMeans use?

A function to plot the silhouette plot is given. More about silhouette plots can be found here [https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_silhouette_analysis.html](https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_silhouette_analysis.html).

**How to plot silhouette:**
```python
silhouette_plot(X_tfidf, cluster_labels)
```

In [42]:
import numpy as np
import matplotlib.cm as cm
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_samples, silhouette_score

def silhouette_plot(X, cluster_labels):
    # Determine the number of clusters
    n_clusters = len(set(cluster_labels))

    # Compute the average silhouette score for all samples
    silhouette_avg = silhouette_score(X, cluster_labels)
    print("Silhouette Avg.:", silhouette_avg)

    # Compute silhouette scores for each sample
    sample_silhouette_values = silhouette_samples(X, cluster_labels)

    # Create a single subplot for the silhouette plot
    fig, (ax1) = plt.subplots(1, 1)

    # Set the y-axis limits based on the number of samples and clusters
    ax1.set_ylim([0, X.shape[0] + (n_clusters + 1) * 100])

    y_lower = 100  # Initial vertical position for the first cluster

    # Loop over each cluster to plot its silhouette values
    for i in range(n_clusters):
        # Get silhouette values for samples in cluster i
        ith_cluster_silhouette_values = sample_silhouette_values[cluster_labels == i]
        ith_cluster_silhouette_values.sort()

        size_cluster_i = ith_cluster_silhouette_values.shape[0]
        y_upper = y_lower + size_cluster_i

        # Choose a color for the cluster
        color = cm.nipy_spectral(float(i) / n_clusters)

        # Fill the silhouette plot for the current cluster
        ax1.fill_betweenx(
            np.arange(y_lower, y_upper),
            0,
            ith_cluster_silhouette_values,
            facecolor=color,
            edgecolor=color,
            alpha=0.7,
        )

        # Label the cluster number
        ax1.text(-0.01, y_lower + 0.5 * size_cluster_i, str(i))

        # Update y_lower for the next cluster
        y_lower = y_upper + 100

    # Add plot titles and labels
    ax1.set_title("The silhouette plot for the various clusters.")
    ax1.set_xlabel("The silhouette coefficient values")
    ax1.set_ylabel("Cluster label")

    # Draw a vertical line for the average silhouette score
    ax1.axvline(x=silhouette_avg, color="red", linestyle="--")

    # Remove y-axis ticks for clarity
    ax1.set_yticks([])

    # Add a super title for the plot
    plt.suptitle(
        f"Silhouette analysis for KMeans clustering on sample data with n_clusters = {n_clusters}",
        fontsize=14,
        fontweight="bold",
    )

    # Display the plot
    plt.show()

In [40]:
from sklearn.cluster import KMeans

# ? your code here

In [41]:

# ? your code here